# FTSE Quality Scorecard

Scores FTSE 100 companies on profitability, balance sheet strength, earnings quality and valuation, and builds the shortlists, backtest and dashboard.

Run the cells top to bottom (**Runtime → Run all** works too). The first run downloads four years of accounts for about 100 companies, so it takes a few minutes.

This is a screen for deciding what to research, not financial advice.

## 1. Get the code and install the libraries

In [ ]:
import os
if not os.path.exists("ftse-quality-scorecard"):
    !git clone -q https://github.com/jathnagulan-cmd/ftse-quality-scorecard.git
else:
    !git -C ftse-quality-scorecard pull -q   # pick up the latest version
%cd ftse-quality-scorecard
!pip install -q -r requirements.txt

## 2. Check the calculations
Runs the tests, which check every formula against hand-worked examples. Everything should pass.

In [ ]:
!python -m pytest -q

## 3. Score every company
Downloads accounts and prices, scores all companies and prints the top 15 plus both shortlists with their strengths and red flags.

In [ ]:
!python run.py

## 4. Look at the results as tables

In [ ]:
import pandas as pd
pd.set_option("display.max_rows", 100)
overall = pd.read_csv("outputs/overall.csv", index_col=0)
print("Long-term shortlist")
display(pd.read_csv("outputs/shortlist_long_term.csv", index_col=0)[["sector", "quality_score", "valuation_score", "overall_score"]])
print("Full ranking")
display(overall.drop(columns=["ticker"]))

Look up any single company's numbers by changing the name below.

In [ ]:
company = "Tesco"
for pillar in ["profitability", "balance_sheet", "earnings_quality", "valuation"]:
    df = pd.read_csv(f"outputs/{pillar}.csv", index_col=0)
    print(f"\n{pillar.replace('_', ' ').title()}")
    print(df.loc[company].drop(labels=[c for c in df.columns if c.endswith("_rank")]).to_string())

## 5. Backtest
Rescores the companies at six past dates using only what was known then, and compares what happened next, including against a FTSE 100 tracker. Takes a couple of minutes.

In [ ]:
!python backtest.py

## 6. Build the dashboard
Creates the same phone-friendly page as your published dashboard, with today's prices, and downloads it so you can open it in your browser.

In [ ]:
!python build_dashboard.py --refresh
from google.colab import files
files.download("outputs/dashboard.html")

## 7. Optional: check which companies are on Trading 212

Needs an API key from the Trading 212 app (Settings → API) with the **metadata** permission only. Never give it trading or withdrawal access.

The key is typed into a hidden box and only kept in memory for this session. It isn't saved in the notebook, so the notebook is safe to share. Skip this cell if you don't need it.

In [ ]:
from getpass import getpass
os.environ["T212_API_KEY"] = getpass("Trading 212 API key: ")
secret = getpass("API secret (press Enter if you don't have one): ")
if secret:
    os.environ["T212_API_SECRET"] = secret
!python check_trading212.py
os.environ.pop("T212_API_KEY", None); os.environ.pop("T212_API_SECRET", None)
!python run.py > /dev/null && echo "Shortlists updated with Trading 212 availability"

## Notes

- Colab runs on Google's computers, so nothing is installed on yours, and anything saved disappears when the session ends. Results you want to keep: download them from the Files panel on the left, or rerun.
- Changes you make to the code here aren't saved to GitHub. To change the project for good, edit it in the repo.